# 04 · Cross-encoder reranking  *(stage 5)* — a negative resultScores the top-50 candidates with a cross-encoder. **~5 minutes on a T4.****Both rerankers made retrieval worse:**| | recall@10 | McNemar p ||---|---|---|| fine-tuned dense, no rerank | **0.3194** | — || + `bge-reranker-base` (278M) | 0.2520 | **0.0003** || + `bge-reranker-v2-m3` (568M) | 0.2877 | 0.068 |### Why it failsThe fine-tuned bi-encoder learned *"is this the same bug"* from 2,000 realduplicate pairs. The cross-encoders are general relevance models trained on websearch, answering *"is this document about this topic"*. Those diverge exactlywhere duplicate detection is hard: the reranker promotes three topically-similarissues above the one that is the actual duplicate.**A 33M domain-adapted bi-encoder beat a 568M general cross-encoder by 3.2points.** Task alignment over parameter count.### Not triedFine-tuning the *cross-encoder* on the same duplicate pairs. The bi-encodergained from domain adaptation; there is no reason a cross-encoder would not.That is the obvious next experiment, recorded rather than claimed.### Upload| file | produced by ||---|---|| `rerank_task.jsonl.gz` | `python -m src.rerank --export-task colab/rerank_task.jsonl.gz` |Only 1.7 MB — it holds the 504 queries, their top-50 candidates, and one copy ofeach candidate's text.

In [ ]:
!pip install -q sentence-transformers

In [ ]:
import torch, gcgc.collect(); torch.cuda.empty_cache()assert torch.cuda.is_available(), "Runtime > Change runtime type > T4 GPU"print(torch.cuda.get_device_name(0),      f"{torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

In [ ]:
import gzip, json, timefrom sentence_transformers import CrossEncoderrecs    = [json.loads(l) for l in gzip.open("rerank_task.jsonl.gz", "rt")]texts   = recs[0]["texts"]      # id -> text, stored once rather than per pairqueries = recs[1:]print(len(queries), "queries x", len(queries[0]["c"]), "candidates")for model_id in ["BAAI/bge-reranker-base", "BAAI/bge-reranker-v2-m3"]:    m = CrossEncoder(model_id, max_length=512)    out, t0 = [], time.time()    for i, r in enumerate(queries):        ids   = [c for c in r["c"] if str(c) in texts]        pairs = [(r["qt"], texts[str(c)]) for c in ids]        s = m.predict(pairs, batch_size=64, show_progress_bar=False)        out.append({"q": r["q"], "c": ids, "s": [float(x) for x in s]})        if i % 100 == 0:            print(model_id.split("/")[-1], i, f"{time.time()-t0:.0f}s")    name = model_id.split("/")[-1]    json.dump(out, gzip.open(f"scores_{name}.json.gz", "wt"))    print("done", name, f"{time.time()-t0:.0f}s")

In [ ]:
from google.colab import filesfiles.download("scores_bge-reranker-base.json.gz")files.download("scores_bge-reranker-v2-m3.json.gz")

### Back on the laptopDrop both files in the project root and score them against the same test split.Scoring reorders each candidate list by the returned scores and recomputesrecall@k / MRR — no model needed locally.